In [2]:
import math
from scipy.optimize import root_scalar

# --- Physical Constants (Legacy Converters) ---
G = 6.67430e-11          # Gravitational constant (m^3 kg^-1 s^-2)
c = 299792458.0          # Speed of light (m/s)
M_sun = 1.98847e30       # Solar mass (kg)
pc_to_m = 3.085677581e16 # 1 parsec in meters

def rs_from_mass(M_kg):
    """Derived system scale Rs from legacy units"""
    return 2.0 * G * M_kg / (c**2)

def exact_will_theta_E_rad(M_kg, D_L_m, D_S_m, beta_p=1.0):
    """
    Numerically solves the implicit WILL RG algebraic equation for the Einstein Ring radius.
    theta_E = 2 * (D_LS / D_S) * arcsin( [kappa_p^2 * (1+beta_p^2)] / [2*beta_p^2 - kappa_p^2*(1+beta_p^2)] )
    where kappa_p^2 = R_s / (D_L * theta_E)
    """
    R_s = rs_from_mass(M_kg)
    D_LS_m = D_S_m - D_L_m

    # Define the implicit residual function to find the root (where residual == 0)
    def will_residual(theta):
        if theta <= 0: return 1.0 # Invalid negative angles

        # 1. Local potential projection evaluated at current theta
        kappa_p_sq = R_s / (D_L_m * theta)

        # 2. Prevent math domain errors for extreme configurations
        b2 = beta_p**2
        num = kappa_p_sq * (1 + b2)
        den = 2 * b2 - num
        if den == 0: return 1.0

        ratio = num / den

        # Check if the ratio is physically captured (e < 1, bounded orbit)
        if ratio > 1.0 or ratio < -1.0: return 1.0

        # 3. The exact unified deflection mapping
        rhs = 2.0 * (D_LS_m / D_S_m) * math.asin(ratio)
        return theta - rhs

    # Provide a starting guess using the weak-field expansion limit
    phase_gradient = (1 + beta_p**2) / (2 * beta_p**2)
    theta_cl_guess = math.sqrt(2.0 * R_s * phase_gradient * D_LS_m / (D_L_m * D_S_m))

    try:
        # Numerically isolate the root (bracketed around the classical guess)
        sol = root_scalar(will_residual, bracket=[theta_cl_guess * 0.1, theta_cl_guess * 1.9], method='brentq')
        return sol.root if sol.converged else float('nan')
    except ValueError:
        return float('nan')

# =====================================================================
# TEST CONFIGURATIONS
# =====================================================================
cases = [
    {
        "name": "Galaxy Lens (Weak Field, Light)",
        "M": 1e12 * M_sun,
        "DL": 1e9 * pc_to_m,    # 1 Gpc
        "DS": 2e9 * pc_to_m,    # 2 Gpc
        "beta": 1.0             # Photon
    },
    {
        "name": "Sgr A* (Strong Field, Light)",
        "M": 4.3e6 * M_sun,
        "DL": 8000 * pc_to_m,   # 8 kpc
        "DS": 8001 * pc_to_m,   # 8 kpc + 1 pc (extreme proximity)
        "beta": 1.0             # Photon
    },
    {
        "name": "Sgr A* (Relativistic Particle)",
        "M": 4.3e6 * M_sun,
        "DL": 8000 * pc_to_m,
        "DS": 8001 * pc_to_m,
        "beta": 0.99            # e.g., Relativistic Neutrino
    }
]

# =====================================================================
# EXECUTION & OUTPUT
# =====================================================================
print("-" * 105)
print("{:<35} | {:<25} | {:<25} | {:<10}".format("Scenario", "Classical Appx (arcsec)", "WILL RG Exact (arcsec)", "Diff (%)"))
print("-" * 105)

for case_item in cases:
    name, M, DL, DS, beta = case_item["name"], case_item["M"], case_item["DL"], case_item["DS"], case_item["beta"]
    DLS = DS - DL
    R_s = rs_from_mass(M)

    # 1. Calculate Classical Approximation (with phase gradient generalized)
    phase_gradient = (1 + beta**2) / (2 * beta**2)
    theta_cl_rad = math.sqrt(2.0 * R_s * phase_gradient * DLS / (DL * DS))
    theta_cl_arcsec = theta_cl_rad * (180.0 / math.pi) * 3600.0

    # 2. Find Exact Root using WILL Relational Geometry
    theta_will_rad = exact_will_theta_E_rad(M, DL, DS, beta_p=beta)
    theta_will_arcsec = theta_will_rad * (180.0 / math.pi) * 3600.0

    # 3. Compare the deviation
    diff = abs(theta_cl_arcsec - theta_will_arcsec) / theta_cl_arcsec * 100
    print("{:<35} | {:<25.8f} | {:<25.8f} | {:<10.4e}".format(name, theta_cl_arcsec, theta_will_arcsec, diff))
print("-" * 105)

---------------------------------------------------------------------------------------------------------
Scenario                            | Classical Appx (arcsec)   | WILL RG Exact (arcsec)    | Diff (%)  
---------------------------------------------------------------------------------------------------------
Galaxy Lens (Weak Field, Light)     | 2.01793205                | 2.01794192                | 4.8877e-04
Sgr A* (Strong Field, Light)        | 0.02339045                | 0.02339576                | 2.2687e-02
Sgr A* (Relativistic Particle)      | 0.02350888                | 0.02351424                | 2.2802e-02
---------------------------------------------------------------------------------------------------------
